In [1]:
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from pathlib import Path

CLEMENTI_DIR = Path("CLEMENTI")
# Load your local Clementi filtered shapefiles
# footpaths = gpd.read_file(f"{CLEMENTI_DIR}/Footpath_Clementi.shp")
footpaths = gpd.read_file(str(CLEMENTI_DIR / "Footpath_Clementi.shp"))
# kerblines = gpd.read_file('clementi_kerblines.shp')
# bridges = gpd.read_file('clementi_bridges.shp')

print(f"Loaded {len(footpaths)} footpath segments.")

Loaded 3960 footpath segments.


In [ ]:
clementi_shp_files = []
# for folder in CLEMENTI_DIR.iterdir():
#     shp_files = list(folder.glob("*.shp"))
#     if not shp_files:
#         print(f"⚠️ No .shp found in {folder.name}, skipping")
#         continue
#     layer = shp_files[0]  # Path to the .shp file
#     print(layer.stem)
#     clementi_shp_files.append(layer.stem)

shp_files = list(CLEMENTI_DIR.glob("*.shp"))
if not shp_files:
    print(f"⚠️ No .shp found in {folder.name}, skipping")
for layer in shp_files:
    clementi_shp_files.append(layer.stem)

print(clementi_shp_files)

['RoadHump_Clementi', 'Gantry_Clementi', 'Railing_Clementi', 'ControllerBox_Clementi', 'PassengerPickupBay_Clementi', 'ArrowMarking_Clementi', 'GuardRail_Clementi', 'BusStop_Clementi', 'Footpath_Clementi', 'RoadSectionLine_Clementi', 'StreetPaint_Clementi', 'RetainingWall_Clementi', 'TrafficSignalAspect_Clementi', 'RoadCrossing_Clementi', 'PedestrainOverheadbridge_Clementi', 'TaxiStop_Clementi', 'DetectorLoop_Clementi', 'LaneMarking_Clementi', 'VehicleOverBridgeUnderpass_Clementi', 'KerbLine_Clementi', 'WordMarking_Clementi', 'CoveredLinkWay_Clementi', 'ParkingZone_Clementi', 'RapidTransitSystemStation_Clementi', 'Bollard_Clementi', 'SpeedRegulatingStrip_Clementi', 'ConvexMirror_Clementi', 'CyclingPathGazette_Clementi', 'LampPost_Clementi']


In [ ]:
clementi_shp_files = [
    "RoadHump_Clementi",
    "Gantry_Clementi",
    "Railing_Clementi",
    "ControllerBox_Clementi",
    "PassengerPickupBay_Clementi",
    "ArrowMarking_Clementi",
    "GuardRail_Clementi",
    "BusStop_Clementi",
    "Footpath_Clementi",
    "RoadSectionLine_Clementi",
    "StreetPaint_Clementi",
    "RetainingWall_Clementi",
    "TrafficSignalAspect_Clementi",
    "RoadCrossing_Clementi",
    "PedestrainOverheadbridge_Clementi",
    "TaxiStop_Clementi",
    "DetectorLoop_Clementi",
    "LaneMarking_Clementi",
    "VehicleOverBridgeUnderpass_Clementi",
    "KerbLine_Clementi",
    "WordMarking_Clementi",
    "CoveredLinkWay_Clementi",
    "ParkingZone_Clementi",
    "RapidTransitSystemStation_Clementi",
    "Bollard_Clementi",
    "SpeedRegulatingStrip_Clementi",
    "ConvexMirror_Clementi",
    "CyclingPathGazette_Clementi",
    "LampPost_Clementi",
]

# Constructing Network Graph


In [2]:
# 2. Construct the Base Network Graph using only Footpaths
G = nx.Graph()

for idx, row in footpaths.iterrows():
    if isinstance(row.geometry, LineString):
        coords = list(row.geometry.coords)
        # Create edges between consecutive coordinates
        for i in range(len(coords) - 1):
            node_a = coords[i]
            node_b = coords[i + 1]

            # Calculate actual distance as baseline weight
            distance = row.geometry.length

            G.add_edge(node_a, node_b, distance=distance, is_bottleneck=False)

In [ ]:
import random

# 3. Identify Step-Drops Ruthlessly Fake the Gaps (Pick 5 random nodes to act as unramped Step-Drops)
all_nodes = list(G.nodes())
fake_step_drops = random.sample(all_nodes, min(5, len(all_nodes)))
for node in fake_step_drops:
    for neighbor in G.neighbors(node):
        G[node][neighbor]["is_bottleneck"] = True

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
print(f"Injected {len(fake_step_drops)} artificial bottlenecks.")

Graph built with 15266 nodes and 12425 edges.
Injected 5 artificial bottlenecks.


In [3]:
# no bottlenecks
all_nodes = list(G.nodes())
print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Graph built with 15266 nodes and 12425 edges.


# Persona Simulation


In the next cell, we apply the core cost formula:

Edge Cost=Distance×Persona Penalty Multiplier


In [7]:
# Sample with Wheelchair Persona
# Define Wheelchair Persona Constraints
PENALTY_MULTIPLIER = 50.0  # Massive penalty for unramped obstacles

# Calculate weights dynamically for every edge
for u, v, data in G.edges(data=True):
    if data["is_bottleneck"]:
        G[u][v]["weight"] = data["distance"] * PENALTY_MULTIPLIER
    else:
        G[u][v]["weight"] = data["distance"] * 1.0

# Trip endpoints (lon, lat) — shapefile coords are (x=longitude, y=latitude)
# START_LON, START_LAT = 103.77270416701658, 1.30098333297992
START_LON, START_LAT = 103.771, 1.303495
END_LON, END_LAT = 103.77165274445558, 1.30783047048090


def nearest_node(G, lon, lat):
    return min(G.nodes(), key=lambda n: (n[0] - lon) ** 2 + (n[1] - lat) ** 2)


start_node = nearest_node(G, START_LON, START_LAT)
end_node = nearest_node(G, END_LON, END_LAT)
print(f"Start snapped to {start_node}")
print(f"End snapped to {end_node}")

try:
    shortest_path = nx.astar_path(G, start_node, end_node, weight="weight")
    print(f"Successfully simulated trip! Path length: {len(shortest_path)} nodes.")
except nx.NetworkXNoPath:
    print("Agent trapped! No valid route found.")

Start snapped to (103.77100352253925, 1.3034817777172645)
End snapped to (103.77167048757657, 1.3078452874656572)
Agent trapped! No valid route found.


# Three.js Build


In [8]:
# Export flat JSON payload for Three.js

import json

output_features = []

for u, v, data in G.edges(data=True):
    line_coords = [list(u), list(v)]
    output_features.append({"coords": line_coords, "isHighCost": data["is_bottleneck"]})

payload = {
    "edges": output_features,
    "start": list(start_node),
    "end": list(end_node),
}

with open("sim_output.json", "w") as f:
    json.dump(payload, f)

print("Flat JSON payload successfully exported for Three.js!")

Flat JSON payload successfully exported for Three.js!
